In [1]:
import glob, os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import cv2
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Device:", device)

Device: cuda


In [3]:
IMG_SIZE = 320
BEST_T = [0.5, 1.5, 2.5, 3.5]

In [4]:
cands = glob.glob('/kaggle/input/*/aptos2019-blindness-detection')
BASE_DIR = cands[0] if cands else '/kaggle/input/aptos2019-blindness-detection'
WEIGHTS = glob.glob('/kaggle/input/notebooks/*/*/best_model_effi.pth')[0]
print("Data:", BASE_DIR)
print("Weights:", WEIGHTS)

Data: /kaggle/input/competitions/aptos2019-blindness-detection
Weights: /kaggle/input/notebooks/suyogkarki1/aptos2019-training/best_model_effi.pth


In [5]:
effinet = models.efficientnet_b3(weights=None)
effinet.classifier = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(effinet.classifier[1].in_features, 1),
)
effinet.load_state_dict(torch.load(WEIGHTS, map_location=device))
effinet = effinet.to(device).eval()
print("Model loaded")

Model loaded


In [6]:
def crop_circle(img, tol=7):
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    mask = gray > tol
    if mask.sum() == 0:
        return img
    ys, xs = np.where(mask)
    return img[ys.min():ys.max()+1, xs.min():xs.max()+1]

def ben_graham(path, size=IMG_SIZE, sigma=10):
    img = cv2.cvtColor(cv2.imread(path), cv2.COLOR_BGR2RGB)
    img = cv2.resize(crop_circle(img), (size, size))
    img = cv2.addWeighted(img, 4, cv2.GaussianBlur(img, (0, 0), sigma), -4, 128)
    return Image.fromarray(img)

In [7]:
test_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

In [8]:
class KaggleTestDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform
    def __len__(self):
        return len(self.df)
    def __getitem__(self, index):
        return self.transform(ben_graham(self.df.iloc[index]['file_path']))

In [9]:
test_df = pd.read_csv(f'{BASE_DIR}/test.csv')
test_df['file_path'] = test_df['id_code'].apply(lambda x: f'{BASE_DIR}/test_images/{x}.png')
test_loader = DataLoader(KaggleTestDataset(test_df, test_transform), batch_size=32, shuffle=False, num_workers=4)

In [10]:
raw = []
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        p = (effinet(images).squeeze(1) + effinet(torch.flip(images, dims=[3])).squeeze(1)) / 2
        raw.extend(p.float().cpu().numpy())

In [11]:
submission = pd.DataFrame({'id_code': test_df['id_code'], 'diagnosis': np.digitize(raw, BEST_T)})
submission.to_csv('/kaggle/working/submission.csv', index=False)
print(submission['diagnosis'].value_counts().sort_index())
print("submission.csv saved:", os.path.exists('/kaggle/working/submission.csv'))

diagnosis
0     285
1     248
2    1185
3     184
4      26
Name: count, dtype: int64
submission.csv saved: True
